In [1]:
import sys
sys.path.append('..')

from matplotlib import pyplot as plt
from src.analysis import break_even_analysis, comparative_statics, stochastic_bivariate_simulation, run_monte_carlo, \
    run_two_way_sensitivity_analysis
from src.config import variable_names
from src.engine import evaluate_chained_models, evaluate_variable_scenario_sweep, evaluate_expected_scenario
from src.models import ProfitModel, CostOfGoodsSoldModel, RevenueModel, FreeCashFlowModel, \
    CapitalExpenditureModel, DepreciationModel, TotalExpenseModel, NetIncomeModel, RoiModel, TotalCostModel, \
    AdvertisingEfficiencyModel
from src.variables import Orders, SellingPrice, PurchasingPrice, ItemsPerOrder, USDToRMB, Rent, TaxRate, \
    CostPerAcquisition, ConversionRate, RenderFee, TravelFee, AdvertisingCost, Expense

from src.visualization import render_break_even_dashboard, get_break_even_dataframe

In [2]:
variables = {variable_names.DEAL_ITEMS_PER_ORDER: ItemsPerOrder(min_value=1, max_value=5),
             variable_names.FINANCE_USD_TO_RMB: USDToRMB(expected_value=6.8, min_value=6.0, max_value=7.5),
             variable_names.DEAL_ORDERS: Orders(min_value=20, max_value=30),
             variable_names.COST_ADVERTISING: AdvertisingCost(min_value=10000, max_value=30000),
             variable_names.COST_CPA: CostPerAcquisition(min_value=12, max_value=36),
             variable_names.COST_CONVERSION_RATE: ConversionRate(min_value=0.04, max_value=0.2),
             variable_names.DEAL_PURCHASING_PRICE: PurchasingPrice(min_value=1000, max_value=2000),
             variable_names.DEAL_SELLING_PRICE: SellingPrice(min_value=3000, max_value=6000)}

cogs_model = CostOfGoodsSoldModel()
cost_model = TotalCostModel()
revenue_model = RevenueModel()
expense_model = TotalExpenseModel()
depreciation_model = DepreciationModel()
capital_expenditure_model = CapitalExpenditureModel()
net_income_model = NetIncomeModel()
profit_model = ProfitModel()
advertising_efficiency_model = AdvertisingEfficiencyModel()

In [3]:
break_even_analysis_report = break_even_analysis(variables, selected_variables=[variable_names.DEAL_ORDERS,
                                                                                variable_names.DEAL_SELLING_PRICE],
                                                 model_pipeline=[advertising_efficiency_model,
                                                                 cogs_model, revenue_model, cost_model,
                                                                 expense_model,
                                                                 depreciation_model, net_income_model,
                                                                 capital_expenditure_model, profit_model],
                                                 output_name=variable_names.PROFIT, goal=0)

print(get_break_even_dataframe(break_even_analysis_report, variable_names.PROFIT))

  Sensitivity Variable  Base (Expected)  BE (Threshold) Safety Margin %
0            └─ Profit     1.263824e+06    1.263824e+06               -
1               Orders     2.500000e+01    2.000000e+01           20.0%
2            └─ Profit     1.263824e+06    8.138235e+05               -
3         SellingPrice     4.500000e+03    3.000000e+03           33.3%


In [4]:
render_break_even_dashboard(break_even_analysis_report, variable_names.PROFIT)

Sensitivity Variable,Base (Expected),BE (Threshold),Safety Margin %
Profit,"¥1,263,824","¥1,263,824",
Orders,25.0,20.0,20.0%
Profit,"¥1,263,824","¥813,824",
SellingPrice,"$4,500","$3,000",33.3%


In [5]:
print(break_even_analysis_report)

[{'BreakEvenVariable': 'Orders', 'feasibility_status': 'ALWAYS_FEASIBLE', 'BreakEvenExpectedVariableValue': 25.0, 'BreakEvenExpectedResult': 1263823.5294117648, 'ThresholdVariableValue': 20.0, 'ThresholdResult': 1263823.5294117648, 'SafetyMarginPercentage': 0.2}, {'BreakEvenVariable': 'SellingPrice', 'feasibility_status': 'ALWAYS_FEASIBLE', 'BreakEvenExpectedVariableValue': 4500.0, 'BreakEvenExpectedResult': 1263823.5294117648, 'ThresholdVariableValue': 3000.0, 'ThresholdResult': 813823.5294117648, 'SafetyMarginPercentage': 0.3333333333333333}]


In [6]:
break_even_analysis_report_higher_goal = break_even_analysis(variables,
                                                             selected_variables=[variable_names.DEAL_ORDERS,
                                                                                 variable_names.DEAL_SELLING_PRICE],
                                                             model_pipeline=[advertising_efficiency_model,
                                                                             cogs_model, revenue_model, cost_model,
                                                                             expense_model,
                                                                             depreciation_model, net_income_model,
                                                                             capital_expenditure_model,
                                                                             profit_model],
                                                             output_name=variable_names.PROFIT, goal=1000000)
print(get_break_even_dataframe(break_even_analysis_report_higher_goal, variable_names.PROFIT))

  Sensitivity Variable  Base (Expected)  BE (Threshold) Safety Margin %
0            └─ Profit     1.263824e+06    1.263824e+06               -
1               Orders     2.500000e+01    2.000000e+01           20.0%
2            └─ Profit     1.263824e+06    1.015864e+06               -
3         SellingPrice     4.500000e+03    3.673469e+03           18.4%


In [7]:
render_break_even_dashboard(break_even_analysis_report_higher_goal, variable_names.PROFIT)

Sensitivity Variable,Base (Expected),BE (Threshold),Safety Margin %
Profit,"¥1,263,824","¥1,263,824",
Orders,25.0,20.0,20.0%
Profit,"¥1,263,824","¥1,015,864",
SellingPrice,"$4,500","$3,673",18.4%


In [8]:
print(break_even_analysis_report_higher_goal)

[{'BreakEvenVariable': 'Orders', 'feasibility_status': 'ALWAYS_FEASIBLE', 'BreakEvenExpectedVariableValue': 25.0, 'BreakEvenExpectedResult': 1263823.5294117648, 'ThresholdVariableValue': 20.0, 'ThresholdResult': 1263823.5294117648, 'SafetyMarginPercentage': 0.2}, {'BreakEvenVariable': 'SellingPrice', 'feasibility_status': 'CROSSOVER_FOUND', 'BreakEvenExpectedVariableValue': 4500.0, 'BreakEvenExpectedResult': 1263823.5294117648, 'ThresholdVariableValue': 3673.4694, 'ThresholdResult': 1015864.3494117648, 'SafetyMarginPercentage': 0.18367346666666667}]


In [9]:
break_even_analysis_report_highest_goal = break_even_analysis(variables,
                                                             selected_variables=[variable_names.COST_ADVERTISING,
                                                                                 variable_names.DEAL_PURCHASING_PRICE,
                                                                                 variable_names.DEAL_SELLING_PRICE,
                                                                                 variable_names.COST_CPA,
                                                                                 variable_names.COST_CONVERSION_RATE,
                                                                                 variable_names.DEAL_ITEMS_PER_ORDER],
                                                             model_pipeline=[advertising_efficiency_model,
                                                                             cogs_model, revenue_model, cost_model,
                                                                             expense_model,
                                                                             depreciation_model, net_income_model,
                                                                             capital_expenditure_model,
                                                                             profit_model],
                                                             output_name=variable_names.PROFIT, goal=1300000)
print(get_break_even_dataframe(break_even_analysis_report_highest_goal, variable_names.PROFIT))

   Sensitivity Variable  Base (Expected)  BE (Threshold) Safety Margin %
0             └─ Profit     1.263824e+06    1.302512e+06               -
1       AdvertisingCost     2.000000e+04    2.061224e+04           -3.1%
2             └─ Profit     1.263824e+06    1.285882e+06               -
3       PurchasingPrice     1.500000e+03    1.000000e+03          -33.3%
4             └─ Profit     1.263824e+06    1.309742e+06               -
5          SellingPrice     4.500000e+03    4.653061e+03           -3.4%
6             └─ Profit     1.263824e+06    1.304366e+06               -
7    CostPerAcquisition     2.400000e+01    2.326530e+01           -3.1%
8             └─ Profit     1.263824e+06    1.316246e+06               -
9        ConversionRate     1.200000e-01    1.249000e-01           -4.1%
10            └─ Profit     1.263824e+06    1.316204e+06               -
11        ItemsPerOrder     3.000000e+00    3.122400e+00           -4.1%


In [10]:
render_break_even_dashboard(break_even_analysis_report_highest_goal, variable_names.PROFIT)

Sensitivity Variable,Base (Expected),BE (Threshold),Safety Margin %
Profit,"¥1,263,824","¥1,302,512",
AdvertisingCost,"¥20,000","¥20,612",-3.1%
Profit,"¥1,263,824","¥1,285,882",
PurchasingPrice,"¥1,500","¥1,000",-33.3%
Profit,"¥1,263,824","¥1,309,742",
SellingPrice,"$4,500","$4,653",-3.4%
Profit,"¥1,263,824","¥1,304,366",
CostPerAcquisition,$24.0,$23.3,-3.1%
Profit,"¥1,263,824","¥1,316,246",
ConversionRate,12.00%,12.49%,-4.1%


In [11]:
print(break_even_analysis_report_highest_goal)

[{'BreakEvenVariable': 'AdvertisingCost', 'feasibility_status': 'CROSSOVER_FOUND', 'BreakEvenExpectedVariableValue': 20000.0, 'BreakEvenExpectedResult': 1263823.5294117648, 'ThresholdVariableValue': 20612.2449, 'ThresholdResult': 1302512.0049308825, 'SafetyMarginPercentage': -0.030612245000000076}, {'BreakEvenVariable': 'PurchasingPrice', 'feasibility_status': 'UNREACHABLE', 'BreakEvenExpectedVariableValue': 1500.0, 'BreakEvenExpectedResult': 1263823.5294117648, 'ThresholdVariableValue': 1000.0, 'ThresholdResult': 1285882.3529411764, 'SafetyMarginPercentage': -0.3333333333333333}, {'BreakEvenVariable': 'SellingPrice', 'feasibility_status': 'CROSSOVER_FOUND', 'BreakEvenExpectedVariableValue': 4500.0, 'BreakEvenExpectedResult': 1263823.5294117648, 'ThresholdVariableValue': 4653.0612, 'ThresholdResult': 1309741.8894117647, 'SafetyMarginPercentage': -0.03401360000000002}, {'BreakEvenVariable': 'CostPerAcquisition', 'feasibility_status': 'CROSSOVER_FOUND', 'BreakEvenExpectedVariableValue': 